# 11 — Daily Data Alignment and Beta-Input Audit

Validate the three daily inputs before any rolling beta is estimated.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(ROOT))

from src.clean_monthly import CleaningConfig, build_clean_monthly_panel
from src.daily_data import read_daily_stock, read_daily_market, read_fama_french_daily_rf, audit_daily_stock, align_market_and_rf, formation_history_availability


## 1. File paths


In [ ]:
DAILY_STOCK = ROOT / 'data/raw/daily_stock_naq_1998_2025_prod_v1.csv.gz'
DAILY_MARKET = ROOT / 'data/raw/daily_crsp_vw_market_1998_2025_v1.csv.gz'
FF_DAILY = ROOT / 'data/raw/F-F_Research_Data_Factors_daily.csv'
MONTHLY = ROOT / 'data/raw/monthly_stock_common_equity_2000_2025_prod_v1.csv.gz'

for p in [DAILY_STOCK, DAILY_MARKET, FF_DAILY, MONTHLY]:
    print(p.name, '->', p.exists())


## 2. Read and audit daily stock file


In [ ]:
daily_stock = read_daily_stock(DAILY_STOCK)
stock_audit = audit_daily_stock(daily_stock)
stock_audit


## 3. Daily return distribution sanity check


In [ ]:
daily_stock['DlyRet'].describe(percentiles=[0.001,0.01,0.05,0.5,0.95,0.99,0.999])


## 4. Read CRSP market and Fama-French RF


In [ ]:
market = read_daily_market(DAILY_MARKET)
rf = read_fama_french_daily_rf(FF_DAILY)
print(market.head())
print(rf.head())


## 5. Align market return and RF


In [ ]:
market_rf, alignment_audit = align_market_and_rf(market, rf)
alignment_audit


## 6. Unit sanity checks


In [ ]:
market_rf[['MarketRet','RF','MarketExcessRet']].describe(percentiles=[0.01,0.5,0.99])


## 7. Formation-history availability on selected months


In [ ]:
clean, _ = build_clean_monthly_panel(MONTHLY, CleaningConfig())
sample_months = ['2000-01','2008-12','2020-12','2025-12']
sample = clean[clean['MthCalDt'].dt.to_period('M').astype(str).isin(sample_months)].copy()
availability = formation_history_availability(sample[['PERMNO','MthCalDt']], daily_stock, lookback_days=252, skip_days=5, min_obs=126)
availability.groupby('MthCalDt').agg(Stocks=('PERMNO','size'), MeanDailyObs=('ValidDailyObs','mean'), MedianDailyObs=('ValidDailyObs','median'), EligibleBeta=('MeetsMinObs','sum'), EligiblePct=('MeetsMinObs','mean'))


## Methodology gate

Proceed only if duplicates are understood, return magnitudes are plausible, market and RF dates align closely, RF units are correct, and representative formation months have adequate daily history.
